# MVTec AD data preparation

Prepare the training and validation image lists for all 15 categories. Each category keeps its own official test images. The final cell saves a split file for each category that the three model owners can share.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

working_dir = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in (working_dir, *working_dir.parents)
        if (folder / "data_tools" / "preparation.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the project folder in VS Code and run the notebook again.")

project_path = str(project_root)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

from data_tools.preparation import (
    category_image_ids,
    create_category_split,
    prepare_image,
    save_category_split,
)

audit_path = project_root / "outputs" / "audit" / "all.json"
if not audit_path.is_file():
    raise FileNotFoundError(
        "Generate outputs/audit/all.json with "
        "python data_tools/audit.py --root data/raw/mvtec-ad --all"
    )

audit = json.loads(audit_path.read_text(encoding="utf-8"))
categories = sorted(audit["categories"])
if audit["status"] != "PASS" or len(categories) != 15 or audit["categories_passed"] != 15:
    raise ValueError("The preparation notebook requires a passing audit for all 15 categories.")

dataset_root = project_root / audit["dataset_root"]
split_dir = project_root / "splits"
print(f"Full dataset audit: {audit['status']} ({len(categories)} categories)")

## Shared splits

For each category, sort the normal training image IDs and shuffle them with seed 42. Take 20% of the images for validation, rounded to the nearest image. Use the rest for training. The official test folders supply the test image IDs.

In [ ]:
manifests = {}
rows = []
for category in categories:
    manifest = create_category_split(dataset_root, category)
    test_ids = category_image_ids(dataset_root, split_dir / f"{category}.json", category, "test")
    manifests[category] = manifest
    rows.append({
        "Category": category,
        "Training": len(manifest["train"]),
        "Validation": len(manifest["validation"]),
        "Official test": len(test_ids),
    })

split_summary = pd.DataFrame(rows)
display(split_summary)

totals = split_summary[["Training", "Validation", "Official test"]].sum()
print(f"Training: {totals['Training']} | Validation: {totals['Validation']} | Official test: {totals['Official test']}")

## Prepared image preview

Every category uses the same full-frame RGB resize to 224 × 224. The shared function returns three color channels with values between 0 and 1. This preview uses a normal bottle training image.

In [ ]:
sample_id = manifests["bottle"]["train"][0]
image_path = dataset_root / sample_id
prepared = prepare_image(image_path)

if prepared.shape != (3, 224, 224) or prepared.dtype != np.float32:
    raise ValueError(f"Unexpected image format: {prepared.shape}, {prepared.dtype}")
if not 0 <= prepared.min() <= prepared.max() <= 1:
    raise ValueError("The prepared image values must be between 0 and 1.")

with Image.open(image_path) as source:
    original = source.convert("RGB")

figure, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
axes[0].imshow(original)
axes[0].set_title(f"Original · {original.width} × {original.height}")
axes[1].imshow(prepared.transpose(1, 2, 0))
axes[1].set_title("Shared input · 224 × 224")
for axis in axes:
    axis.axis("off")
plt.show()
plt.close(figure)

print(f"Example: {sample_id}")
print(f"Shape: {prepared.shape}, type: {prepared.dtype}")

## Save the splits for the team

Run the cell below to save the category files in `splits/`. Each teammate uses the same saved IDs for a given category. A rerun accepts an identical split and flags a conflicting file for review.

In [ ]:
for category, manifest in manifests.items():
    save_category_split(split_dir / f"{category}.json", manifest)

print(f"Splits ready: {len(manifests)} category files in {split_dir.relative_to(project_root).as_posix()}/")